# 🔍 Fake News Detection using NLP and Machine Learning

This notebook demonstrates a complete pipeline for detecting fake news articles using Natural Language Processing (NLP) and Machine Learning techniques.

**Pipeline Steps:**
1. Data Loading & Exploration
2. Text Preprocessing (NLP)
3. TF-IDF Feature Extraction
4. Model Training (Naive Bayes + Logistic Regression)
5. Model Evaluation
6. Visualizations
7. Live Prediction

## Step 0: Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import re
import string

print('All libraries imported successfully!')
print(f'Pandas: {pd.__version__}')
print(f'NumPy: {np.__version__}')

---
## Step 1: Data Loading & Exploration

We load the **Fake** and **True** news datasets from CSV files and combine them into a single DataFrame.

In [ ]:
# Load datasets
fake_df = pd.read_csv('data/Fake.csv')
true_df = pd.read_csv('data/True.csv')

print(f'Fake news articles: {len(fake_df):,}')
print(f'Real news articles: {len(true_df):,}')
print(f'Total articles    : {len(fake_df) + len(true_df):,}')

In [ ]:
# Add labels: 0 = Fake, 1 = Real
fake_df['label'] = 0
true_df['label'] = 1

# Combine datasets
df = pd.concat([fake_df, true_df], axis=0, ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nCombined dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
# Dataset information
print('Dataset Info:')
print(f'  Total samples    : {len(df):,}')
print(f'  Fake news (0)    : {len(df[df["label"] == 0]):,}')
print(f'  Real news (1)    : {len(df[df["label"] == 1]):,}')
print(f'  Missing values   :')
print(df.isnull().sum())
print(f'\n  Avg text length  : {df["text"].str.len().mean():.0f} characters')

In [ ]:
# Visualize label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#E15759', '#4E79A7']
labels = ['Fake News', 'Real News']
counts = df['label'].value_counts().sort_index()

# Bar chart
bars = axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[0].set_title('Label Distribution', fontweight='bold', fontsize=14)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Label Proportion', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.savefig('outputs/label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 2: Text Preprocessing (NLP)

Apply the following NLP preprocessing steps:
- Convert to **lowercase**
- Remove **URLs**, **HTML tags**, **punctuation**, **numbers**
- **Tokenize** text
- Remove **stopwords**
- Apply **stemming** (Porter Stemmer)

In [ ]:
# Initialize NLP tools
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    """Apply all preprocessing steps to a text string."""
    if not isinstance(text, str):
        return ''
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords & apply stemming
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words]
    
    return ' '.join(tokens)

print('Preprocessing function defined!')

In [ ]:
# Combine title + text and apply preprocessing
df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

print('Applying text preprocessing (this may take a few minutes)...')
df['cleaned_content'] = df['content'].apply(clean_text)
print(f'Preprocessing complete! Processed {len(df):,} articles.')

In [ ]:
# Show example: before vs after preprocessing
idx = 0
print('=== BEFORE Preprocessing ===')
print(df['content'].iloc[idx][:300])
print('\n=== AFTER Preprocessing ===')
print(df['cleaned_content'].iloc[idx][:300])

---
## Step 3: TF-IDF Feature Extraction

Convert the preprocessed text into numerical features using **TF-IDF (Term Frequency - Inverse Document Frequency)** vectorization.

In [ ]:
# Split data into train and test sets (80/20)
X = df['cleaned_content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train):,} articles')
print(f'Testing set : {len(X_test):,} articles')

In [ ]:
# Create TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

# Fit on training data and transform
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'TF-IDF matrix shape (test) : {X_test_tfidf.shape}')
print(f'Number of features         : {X_train_tfidf.shape[1]:,}')

---
## Step 4: Model Training

Train two classification models:
1. **Multinomial Naive Bayes** - Probabilistic classifier well-suited for text classification
2. **Logistic Regression** - Linear model for binary classification

In [ ]:
# Train Naive Bayes
print('Training Naive Bayes...')
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)
print('  Naive Bayes training complete!')

# Train Logistic Regression
print('\nTraining Logistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr_model.fit(X_train_tfidf, y_train)
print('  Logistic Regression training complete!')

In [ ]:
# Save models
import joblib
os.makedirs('models', exist_ok=True)

joblib.dump(nb_model, 'models/naive_bayes.joblib')
joblib.dump(lr_model, 'models/logistic_regression.joblib')
joblib.dump(tfidf, 'models/tfidf_vectorizer.joblib')

print('Models and vectorizer saved to models/ directory!')

---
## Step 5: Model Evaluation

Evaluate both models using:
- **Accuracy** - Overall correctness
- **Precision** - How many predicted positives are actually positive
- **Recall** - How many actual positives are correctly identified
- **F1 Score** - Harmonic mean of precision and recall

In [ ]:
# Evaluate both models
models = {'Naive Bayes': nb_model, 'Logistic Regression': lr_model}
results = {}

for name, model in models.items():
    y_pred = model.predict(X_test_tfidf)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cm = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'accuracy': acc, 'precision': prec,
        'recall': rec, 'f1_score': f1,
        'confusion_matrix': cm, 'predictions': y_pred
    }
    
    print(f'\n{"="*55}')
    print(f'  {name} - Results')
    print(f'{"="*55}')
    print(f'  Accuracy  : {acc:.4f} ({acc*100:.2f}%)')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1 Score  : {f1:.4f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Fake", "Real"])}')

---
## Step 6: Visualizations

Generate insightful charts to visualize model performance.

In [ ]:
# Model Accuracy Comparison
os.makedirs('outputs', exist_ok=True)

model_names = list(results.keys())
accuracies = [results[n]['accuracy'] * 100 for n in model_names]
colors = ['#4E79A7', '#E15759']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, accuracies, color=colors, edgecolor='white', linewidth=1.5, width=0.5)

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', va='bottom', fontweight='bold', fontsize=13)

ax.set_title('Model Accuracy Comparison', fontweight='bold', fontsize=16, pad=15)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_ylim(0, 105)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (name, res) in enumerate(results.items()):
    sns.heatmap(res['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'],
                ax=axes[i], linewidths=1, linecolor='white',
                annot_kws={'size': 16, 'weight': 'bold'})
    axes[i].set_title(f'Confusion Matrix - {name}', fontweight='bold', fontsize=14, pad=10)
    axes[i].set_xlabel('Predicted Label', fontsize=12)
    axes[i].set_ylabel('True Label', fontsize=12)

plt.tight_layout()
plt.savefig('outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Metrics Comparison (Grouped Bar Chart)
metric_names = ['accuracy', 'precision', 'recall', 'f1_score']
display_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

x = np.arange(len(display_names))
width = 0.3

fig, ax = plt.subplots(figsize=(12, 6))

for i, name in enumerate(model_names):
    values = [results[name][m] * 100 for m in metric_names]
    offset = (i - 0.5) * width
    bars = ax.bar(x + offset, values, width, label=name, color=colors[i], edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Model Performance Comparison', fontweight='bold', fontsize=16, pad=15)
ax.set_ylabel('Score (%)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(display_names, fontsize=12)
ax.set_ylim(0, 108)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance (Logistic Regression)
feature_names = tfidf.get_feature_names_out()
importance = lr_model.coef_[0]

top_n = 20
top_positive_idx = np.argsort(importance)[-top_n:]
top_negative_idx = np.argsort(importance)[:top_n]

top_idx = np.concatenate([top_negative_idx, top_positive_idx])
top_features = feature_names[top_idx]
top_values = importance[top_idx]

fig, ax = plt.subplots(figsize=(12, 10))
bar_colors = ['#E15759' if v < 0 else '#4E79A7' for v in top_values]
ax.barh(range(len(top_features)), top_values, color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=10)
ax.set_title(f'Top {top_n} Features - Logistic Regression\n(Red = Fake indicators, Blue = Real indicators)',
             fontweight='bold', fontsize=14, pad=15)
ax.set_xlabel('Feature Importance', fontsize=13)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7: Live Prediction

Test the model with custom news text to see if it classifies correctly.

In [ ]:
def predict_news(text, model=lr_model, vectorizer=tfidf):
    """Predict whether a news article is real or fake."""
    # Preprocess
    cleaned = clean_text(text)
    
    # Vectorize
    text_tfidf = vectorizer.transform([cleaned])
    
    # Predict
    prediction = model.predict(text_tfidf)[0]
    proba = model.predict_proba(text_tfidf)[0]
    confidence = max(proba) * 100
    
    label = 'REAL ✅' if prediction == 1 else 'FAKE ⚠️'
    
    print(f'\n{"="*50}')
    print(f'  Prediction : {label}')
    print(f'  Confidence : {confidence:.2f}%')
    print(f'{"="*50}')
    
    return label, confidence

In [ ]:
# Test with a REAL news sample
real_sample = """
WASHINGTON (Reuters) - The U.S. Senate passed a sweeping tax overhaul bill early Saturday, 
moving President Donald Trump and Republicans a big step closer to enacting the most ambitious 
rewrite of the tax code in three decades. The vote was 51-49, with only one Republican, 
Bob Corker of Tennessee, breaking ranks to vote against a bill that would add an estimated 
$1.4 trillion to the $20 trillion national debt over 10 years.
"""

print('Testing with a REAL news article:')
predict_news(real_sample)

In [ ]:
# Test with a FAKE news sample
fake_sample = """
BREAKING: Scientists discover that the moon is actually made of cheese! NASA has been 
hiding this fact for decades. Multiple whistleblowers from within the agency have come 
forward with shocking evidence that the Apollo missions were actually cheese-collecting 
expeditions. The government has been stockpiling lunar cheese in secret underground 
bunkers across the country.
"""

print('Testing with a FAKE news article:')
predict_news(fake_sample)

---
## Summary

| Model | Accuracy | Precision | Recall | F1 Score |
|-------|----------|-----------|--------|----------|
| Naive Bayes | See above | See above | See above | See above |
| Logistic Regression | See above | See above | See above | See above |

### Key Takeaways:
- Both models achieve **high accuracy** (~90%+) on the fake news detection task
- **TF-IDF** with bigrams effectively captures distinguishing features
- **Logistic Regression** typically performs slightly better for this type of text classification
- The preprocessing pipeline (stopword removal, stemming) is crucial for good performance

### Next Steps:
- Try **Deep Learning** models (BERT, LSTM) for potentially better performance
- Detect fake news from **social media posts**
- Build a **real-time web application** using Flask (already provided in `app.py`)